> 在线讲义：[第 46 章 · KV 缓存的分配、复用与释放：分页、共享、留不留](https://wikieden.github.io/ai-infra-from-constraints/lessons/46-kv-cache.html) · 本书《从约束推导 AI Infra》

# 第 46 章 · KV 缓存的分配、复用与释放：分页、共享、留不留

配套讲义：仓库里 `ai-infra/notes/lessons/46-kv-cache.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



分页与前缀共享的块数、写时复制、16 条共享前缀长请求的容量、前缀保留的期望收益取自 AII 第 8 章 §8.3。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math

**函数 `paged`**：(分配的 token 容量, 未用槽位)。

In [ ]:
def paged(lengths: list[int], p: int) -> tuple[int, int]:
    """(分配的 token 容量, 未用槽位)。"""
    alloc = sum(math.ceil(L / p) * p for L in lengths)
    return alloc, alloc - sum(lengths)

**函数 `shared_prefix_mib`**：小工具。

In [ ]:
def shared_prefix_mib(b: int, prefix: float = 864, private: float = 324) -> float:
    return prefix + b * private

**函数 `max_shared`**：小工具。

In [ ]:
def max_shared(budget: float = 12 * 1024, prefix: float = 864, private: float = 324) -> int:
    return int((budget - prefix) // private)

**函数 `keep_value`**：V_h = p(Tr − Tf) − Tm（ms）。

In [ ]:
def keep_value(p: float, Tr: float, Tf: float, Tm: float) -> float:
    """V_h = p(Tr − Tf) − Tm（ms）。"""
    return p * (Tr - Tf) - Tm

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：分页")
L = [9, 13, 5, 15]
print(f"  整段预留 {4 * 16}（未用 {64 - sum(L)}）")
for p in (2, 4, 8):
    a, w = paged(L, p)
    print(f"  每块 {p}：分配 {a}，未用 {w}")
print(f"  A、B 共享前 8 个 token：{paged(L, 4)[0] - 8} 个")

In [ ]:
print("\n第二步：共享与写时复制")
refs, phys = 4 * (95 + 9), 95 + 4 * 9
print(f"  四分支：引用 {refs}，物理块 {phys}，少 {1 - phys / refs:.0%}")
print(f"  16 条 8K 长请求：独立 {16 * 1188:,} MiB，共享前缀 {shared_prefix_mib(16):,.0f} MiB（{shared_prefix_mib(16) / 1024:.2f} GiB）")
print(f"  12 GiB 最多：独立 {12 * 1024 // 1188} 条，共享 {max_shared()} 条")

In [ ]:
print("\n第三步：前缀值不值得留（RTX PRO 6000）")
gen5, gen4 = 0.906e9 / 64e9 * 1e3, 0.906e9 / 32e9 * 1e3
va = keep_value(0.5, 307.9, gen5, gen5)
print(f"  A（6144 token，864 MiB）：取回 {gen5:.1f} ms，命中 50% 净省 {va:.1f} ms，门槛 {gen5 / (307.9 - gen5):.1%}")
print(f"  PCIe Gen4：取回 {gen4:.1f} ms，门槛 {gen4 / (307.9 - gen4):.1%}")
vb = keep_value(0.5, 94.7, 4.7, 4.7)
print(f"  B（2048 token，288 MiB）：净省 {vb:.1f} ms，三个 {3 * vb:.1f} ms；每 MiB A {va / 864:.3f}、B {vb / 288:.3f} ms")

## 3. 代码里才看得到的

1. 6144 个 token 的前缀约 0.906 GB（864 MiB），PCIe Gen5 64 GB/s 取回 14.2 ms，维护按一次换出算同样 14.2 ms。
2. 重算时间 307.9 ms 是原文按 96.5 TFLOP、62% MFU 给的值，代码直接用。
3. 共享后容量上限是 $\lfloor (12288 - 864)/324 \rfloor = 35$。

## 4. 自己改一改

- 给 `paged` 加前缀共享：输入每条的共同前缀长度，算节省多少块。
- 用 `keep_value` 画 A、B 在命中率 0–100% 时的曲线，找 A 不如三个 B 的命中率。
- 模拟一个容量固定的 LRU 前缀缓存，看 12 轮对话的命中数。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 活页册页码定顺序，纸夹哪都行；只有最后一页会空；
> 合租的书记人数，最后一人走才还；要做笔记先复印；
> 抄写接龙只能从头接，开头改一字后面全重抄；
> 剩菜值不值得留：下顿吃的概率乘省下的工夫，再按冰箱格子算。